# Entrenamiento Modelo CNN Multi-Horizonte

Este notebook entrena un modelo CNN para predecir la probabilidad de incumplimiento de créditos a 18 horizontes de tiempo.

## Entradas
- `output/datasets/datos_preprocesados.csv` (generado por EDA.ipynb)

## Salidas
- Modelo CNN entrenado
- Scaler para normalización
- Archivo de configuración con métricas

## Criterio de éxito
- El modelo se entrena sin errores
- Se guardan todos los artefactos requeridos
- Se imprime resumen de métricas

In [1]:
import json
import os
import sys
from datetime import datetime

import joblib
import mlflow
import mlflow.tensorflow
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)

sys.path.insert(0, '..')
from sklearn.preprocessing import MinMaxScaler
from src.ts_cnn.dashboard import build_all_plots
from tensorflow.keras import Model, layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


/home/ovelez/Documentos/cursos/Maestria/IADegreeThesis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1784172440.278927  393670 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784172440.279277  393670 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1784172440.315917  393670 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler fla

## Configuración del notebook

In [2]:
# Configuración de rutas
PATH_SALIDA = "output"
PATH_INPUT = f"{PATH_SALIDA}/datasets/datos_preprocesados.csv"
PATH_OUTPUT = f"{PATH_SALIDA}/modelos_cnn"
os.makedirs(PATH_OUTPUT, exist_ok=True)

# Configuración del modelo
VENTANA_CNN = 6  # Meses de historial para la secuencia
MAX_HORIZONTE = 18  # Meses futuros a predecir
EPOCHS = 100
BATCH_SIZE = 32
PATIENCE = 10  # Early stopping

print("Configuración:")
print(f"- Ventana CNN: {VENTANA_CNN} meses")
print(f"- Máximo horizonte: {MAX_HORIZONTE} meses")
print(f"- Épocas máximas: {EPOCHS}")
print(f"- Batch size: {BATCH_SIZE}")

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("jupy_entrenamiento_cnn")

Configuración:
- Ventana CNN: 6 meses
- Máximo horizonte: 18 meses
- Épocas máximas: 100
- Batch size: 32


<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1783825854941, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1783825854941, lifecycle_stage='active', name='jupy_entrenamiento_cnn', tags={}, trace_location=None, workspace='default'>

## Carga de datos

In [3]:
# Cargar datos preprocesados
df = pd.read_csv(PATH_INPUT)
df['mes'] = pd.to_datetime(df['mes'])

print(f"Datos cargados: {len(df):,} registros")
print(f"Columnas: {len(df.columns)}")
print("\nDistribución crisis_flag:")
print(df['crisis_flag'].value_counts())
print("\nPrimeras filas:")
df.head()

Datos cargados: 45,077 registros
Columnas: 29

Distribución crisis_flag:
crisis_flag
0    39660
1     5417
Name: count, dtype: int64

Primeras filas:


,mes,riesgo,sector,codigo_sucursal,bloque_id,num_creditos,monto_total,monto_promedio,plazo_promedio,tasa_interes_promedio,...,num_clientes_unicos,tasa_judicial,tasa_cierre,tasa_mora_90,desviacion_montos,coef_variacion_montos,creditos_por_cliente,tasa_crecimiento_creditos,tasa_crecimiento_monto,crisis_flag
0,2016-05-01,2,13,1,2_13_1,72,2160000.0,2160000.0,72.0,10.5,...,1,0.00,100.0,0.0,0.00,0.00,72.0,NaN,NaN,0
1,2017-12-01,2,13,1,2_13_1,84,12600000.0,12600000.0,84.0,10.5,...,1,0.00,100.0,0.0,0.00,0.00,84.0,250.00,3938.46,0
2,2018-08-01,2,13,1,2_13_1,60,960000.0,960000.0,60.0,10.5,...,1,0.00,100.0,0.0,0.00,0.00,60.0,400.00,1500.00,0
3,2019-05-01,2,13,1,2_13_1,60,15420000.0,15420000.0,60.0,11.0,...,1,0.00,100.0,0.0,0.00,0.00,60.0,1400.00,19175.00,0
4,2019-08-01,2,13,1,2_13_1,74,1300000.0,1300000.0,51.3,10.5,...,2,81.08,100.0,0.0,15773.29,1.21,37.0,54.17,35.42,1


## Preprocesamiento y features

In [4]:
def preprocesar_datos(df):
    """
    Preprocesa datos para CNN.
    - Remueve features con cero varianza.
    - Recorta outliers al percentil 1-99.
    """
    df_features = df.copy()
    df_features['mes'] = pd.to_datetime(df_features['mes'])
    
    features_numericas = [
        'num_creditos', 'monto_total', 'monto_promedio', 'plazo_promedio',
        'tasa_interes_promedio', 'saldo_promedio', 'total_costo_judicial',
        'total_gestion_cobro', 'total_notificaciones', 'tot_dias_mora_promedio',
        'tot_num_moras_promedio', 'mora_promedio', 'creditos_judiciales',
        'creditos_cerrados', 'tasa_judicial', 'tasa_cierre',
        'tasa_mora_90', 'creditos_por_cliente', 'coef_variacion_montos',
        'tasa_crecimiento_creditos', 'tasa_crecimiento_monto',
    ]
    
    features_existentes = [f for f in features_numericas if f in df_features.columns]
    
    # Remover features con cero varianza (siempre el mismo valor)
    features_limpias = []
    for f in features_existentes:
        if df_features[f].std() > 0:
            features_limpias.append(f)
        else:
            print(f"Removida feature sin varianza: {f}")
    
    # Recortar outliers al percentil 1-99
    for f in features_limpias:
        q01 = df_features[f].quantile(0.01)
        q99 = df_features[f].quantile(0.99)
        df_features[f] = df_features[f].clip(lower=q01, upper=q99)
    
    df_features = df_features.sort_values(['bloque_id', 'mes'])
    
    print(f"Features seleccionadas: {len(features_limpias)}")
    print(f"Bloques unicos: {df_features['bloque_id'].nunique()}")
    
    return df_features, features_limpias

In [5]:
df_features, features_numericas = preprocesar_datos(df)
print(f"\nFeatures para el modelo ({len(features_numericas)}):")
print(features_numericas)

Removida feature sin varianza: total_gestion_cobro
Removida feature sin varianza: tot_num_moras_promedio
Removida feature sin varianza: tasa_cierre
Features seleccionadas: 18
Bloques unicos: 1158

Features para el modelo (18):
['num_creditos', 'monto_total', 'monto_promedio', 'plazo_promedio', 'tasa_interes_promedio', 'saldo_promedio', 'total_costo_judicial', 'total_notificaciones', 'tot_dias_mora_promedio', 'mora_promedio', 'creditos_judiciales', 'creditos_cerrados', 'tasa_judicial', 'tasa_mora_90', 'creditos_por_cliente', 'coef_variacion_montos', 'tasa_crecimiento_creditos', 'tasa_crecimiento_monto']


## Creación de secuencias temporales

In [6]:
def crear_secuencias_cnn(df, bloque_id, features, target, ventana, max_horizonte):
    """
    Genera secuencias temporales para entrenamiento de una CNN multi-horizonte.
    Retorna X, y y las fechas de prediccion para split temporal.
    """
    df_bloque = df[df['bloque_id'] == bloque_id].sort_values('mes')
    
    if len(df_bloque) < ventana + max_horizonte:
        return None, None, None
    
    X_sequences = []
    y_sequences = []
    fechas = []
    
    for i in range(len(df_bloque) - ventana - max_horizonte + 1):
        X_seq = df_bloque[features].iloc[i:i+ventana].values
        y_seq = []
        
        for h in range(1, max_horizonte + 1):
            if i + ventana + h - 1 < len(df_bloque):
                y_val = df_bloque[target].iloc[i + ventana + h - 1]
                y_seq.append(y_val)
            else:
                y_seq.append(0)
        
        X_sequences.append(X_seq)
        y_sequences.append(y_seq)
        fechas.append(df_bloque['mes'].iloc[i + ventana - 1])
    
    return np.array(X_sequences), np.array(y_sequences), np.array(fechas)

In [7]:
# Generar secuencias para todos los bloques
X_all = []
y_all = []
fechas_all = []
bloques_validos = []

for bloque in df_features['bloque_id'].unique():
    X_seq, y_seq, fechas_seq = crear_secuencias_cnn(
        df_features, bloque, features_numericas, 'crisis_flag',
        VENTANA_CNN, MAX_HORIZONTE
    )
    if X_seq is not None and len(X_seq) > 0:
        X_all.extend(X_seq)
        y_all.extend(y_seq)
        fechas_all.extend(fechas_seq)
        bloques_validos.append(bloque)

X_cnn = np.array(X_all)
y_cnn = np.array(y_all)
fechas_cnn = np.array(fechas_all)

print(f"Secuencias generadas:")
print(f"- X shape: {X_cnn.shape}")
print(f"- y shape: {y_cnn.shape}")
print(f"- Fechas: {fechas_cnn.min()} a {fechas_cnn.max()}")
print(f"- Bloques válidos: {len(bloques_validos)}")

Secuencias generadas:
- X shape: (25434, 6, 18)
- y shape: (25434, 18)
- Fechas: 2016-06-01 00:00:00 a 2024-06-01 00:00:00
- Bloques válidos: 680


## Normalización y división de datos

In [8]:
# Validación antes de normalizar
if X_cnn.size == 0 or X_cnn.ndim != 3:
    min_meses = VENTANA_CNN + MAX_HORIZONTE
    meses_por_bloque = df_features.groupby('bloque_id').size()
    resumen = meses_por_bloque.describe().to_dict()
    
    raise ValueError(
        "No se generaron secuencias CNN. "
        f"Se requieren al menos {min_meses} meses por bloque "
        f"(VENTANA_CNN={VENTANA_CNN}, MAX_HORIZONTE={MAX_HORIZONTE}). "
        f"Bloques válidos: {len(bloques_validos)}. "
        f"Resumen meses/bloque: {resumen}"
    )

# Normalizar features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_cnn.reshape(-1, X_cnn.shape[-1])).reshape(X_cnn.shape)

# Ordenar por mes_prediccion para split temporal (sin data leakage)
sort_idx = np.argsort(fechas_cnn)
X_scaled = X_scaled[sort_idx]
y_cnn = y_cnn[sort_idx]
fechas_cnn = fechas_cnn[sort_idx]
print(f'Fechas de prediccion: {fechas_cnn.min()} a {fechas_cnn.max()}')

# Split temporal 70/30
split_idx = int(len(X_scaled) * 0.7)
X_train, X_test = X_scaled[:split_idx], X_scaled[split_idx:]

# Preparar listas de targets por horizonte
y_train_list = [y_cnn[:split_idx, i] for i in range(MAX_HORIZONTE)]
y_test_list = [y_cnn[split_idx:, i] for i in range(MAX_HORIZONTE)]

# Asegurar shapes correctos
for i in range(len(y_train_list)):
    y_train_list[i] = np.asarray(y_train_list[i]).reshape(-1,)
for i in range(len(y_test_list)):
    y_test_list[i] = np.asarray(y_test_list[i]).reshape(-1,)

print("División de datos:")
print(f"- Train: {X_train.shape[0]:,} muestras")
print(f"- Test: {X_test.shape[0]:,} muestras")
print(f"- Features: {X_train.shape[2]}")
print(f"- Horizontes: {MAX_HORIZONTE}")

Fechas de prediccion: 2016-06-01 00:00:00 a 2024-06-01 00:00:00
División de datos:
- Train: 17,803 muestras
- Test: 7,631 muestras
- Features: 18
- Horizontes: 18


## Construcción del modelo CNN

In [9]:
def crear_modelo_cnn_multioutput(input_shape, num_horizontes):
    """
    Crea un modelo CNN con múltiples salidas para predecir crisis en varios horizontes.
    """
    inputs = layers.Input(shape=input_shape)
    
    # Capa convolucional 1
    x = layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    # Capa convolucional 2
    x = layers.Conv1D(filters=128, kernel_size=3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.3)(x)
    
    # Capas densas
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    
    # Salidas múltiples (una por horizonte)
    outputs = []
    for i in range(num_horizontes):
        output = layers.Dense(1, activation='sigmoid', name=f'horizonte_{i+1}')(x)
        outputs.append(output)
    
    model = Model(inputs=inputs, outputs=outputs)
    
    # Métricas por horizonte
    metrics_dict = {
        f'horizonte_{i+1}': [
            'accuracy',
            tf.keras.metrics.Precision(name=f'precision_{i+1}'),
            tf.keras.metrics.Recall(name=f'recall_{i+1}'),
        ]
        for i in range(num_horizontes)
    }
    
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=metrics_dict
    )
    
    return model

In [10]:
# Crear modelo
input_shape = X_train.shape[1:]
modelo_cnn = crear_modelo_cnn_multioutput(input_shape, MAX_HORIZONTE)

# Resumen del modelo
modelo_cnn.summary()

E0000 00:00:1784172450.811504  393670 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 6, 18)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 6, 64)     │      3,520 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 6, 64)     │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 6, 64)     │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 6, 128)    │     24,704 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 6, 128)    │        512 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 3, 128)    │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 3, 128)    │          0 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 384)       │          0 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     49,280 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_1 (Dense) │ (None, 1)         │         65 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_2 (Dense) │ (None, 1)         │         65 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_3 (Dense) │ (None, 1)         │         65 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_4 (Dense) │ (None, 1)         │         65 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_5 (Dense) │ (None, 1)         │         65 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_6 (Dense) │ (None, 1)         │         65 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_7 (Dense) │ (None, 1)         │         65 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ horizonte_8 (Dense) │ (None, 1)         │         65 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 88,210 (344.57 KB)

 Trainable params: 87,570 (342.07 KB)

 Non-trainable params: 640 (2.50 KB)

## Entrenamiento del modelo

In [11]:
# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        os.path.join(PATH_OUTPUT, 'best_model_cnn.keras'),
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

# División train/validation (80/20)
val_frac = 0.2
n_samples = X_train.shape[0]
val_size = int(n_samples * val_frac)
train_end = n_samples - val_size

X_train_final = X_train[:train_end]
X_val = X_train[train_end:]
y_train_final = [arr[:train_end] for arr in y_train_list]
y_val = [arr[train_end:] for arr in y_train_list]

# Sample weights para manejar desbalanceo de clases
# Keras no soporta class_weight con multi-output, usamos sample_weight
sample_weight_train = []
sample_weight_val = []
for h in range(MAX_HORIZONTE):
    y_h_train = y_train_final[h]
    n_pos = int(np.sum(y_h_train == 1))
    n_neg = int(np.sum(y_h_train == 0))
    total = n_pos + n_neg
    w_0 = total / (2.0 * n_neg) if n_neg > 0 else 1.0
    w_1 = total / (2.0 * n_pos) if n_pos > 0 else 1.0

    sw_train = np.where(y_h_train == 1, w_1, w_0).astype(np.float32)
    y_h_val = y_val[h]
    n_pos_v = int(np.sum(y_h_val == 1))
    n_neg_v = int(np.sum(y_h_val == 0))
    total_v = n_pos_v + n_neg_v
    w_0_v = total_v / (2.0 * n_neg_v) if n_neg_v > 0 else 1.0
    w_1_v = total_v / (2.0 * n_pos_v) if n_pos_v > 0 else 1.0
    sw_val = np.where(y_h_val == 1, w_1_v, w_0_v).astype(np.float32)

    sample_weight_train.append(sw_train)
    sample_weight_val.append(sw_val)
    print(f'  Horizonte {h+1}: crisis_train={n_pos} ({n_pos/total*100:.1f}%), w_1={w_1:.2f}')

mlflow.start_run(
    run_name=f"entrenamiento_cnn_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)
mlflow.log_param('ventana_cnn', VENTANA_CNN)
mlflow.log_param('max_horizonte', MAX_HORIZONTE)
mlflow.log_param('epochs', EPOCHS)
mlflow.log_param('batch_size', BATCH_SIZE)
mlflow.log_param('num_features', len(features_numericas))
print('\nIniciando entrenamiento con sample weights...')
historia = modelo_cnn.fit(
    X_train_final,
    y_train_final,
    validation_data=(X_val, y_val, sample_weight_val),
    sample_weight=sample_weight_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

# Calculamos métricas de test antes de registrarlas en MLflow
y_pred_proba_list = modelo_cnn.predict(X_test, verbose=0)
y_pred_1m = (y_pred_proba_list[0] > 0.5).astype(int).flatten()
y_test_1m = y_test_list[0]

test_acc = accuracy_score(y_test_1m, y_pred_1m)
test_prec = precision_score(y_test_1m, y_pred_1m, zero_division=0)
test_recall = recall_score(y_test_1m, y_pred_1m, zero_division=0)

mlflow.tensorflow.log_model(modelo_cnn, 'modelo')
mlflow.log_metric('final_accuracy', float(test_acc))
mlflow.log_metric('final_precision', float(test_prec))
mlflow.log_metric('final_recall', float(test_recall))
mlflow.end_run()

  Horizonte 1: crisis_train=1044 (7.3%), w_1=6.82
  Horizonte 2: crisis_train=1077 (7.6%), w_1=6.61
  Horizonte 3: crisis_train=1100 (7.7%), w_1=6.47
  Horizonte 4: crisis_train=1124 (7.9%), w_1=6.34
  Horizonte 5: crisis_train=1160 (8.1%), w_1=6.14
  Horizonte 6: crisis_train=1197 (8.4%), w_1=5.95
  Horizonte 7: crisis_train=1234 (8.7%), w_1=5.77
  Horizonte 8: crisis_train=1276 (9.0%), w_1=5.58
  Horizonte 9: crisis_train=1346 (9.5%), w_1=5.29
  Horizonte 10: crisis_train=1376 (9.7%), w_1=5.18
  Horizonte 11: crisis_train=1412 (9.9%), w_1=5.04
  Horizonte 12: crisis_train=1450 (10.2%), w_1=4.91
  Horizonte 13: crisis_train=1453 (10.2%), w_1=4.90
  Horizonte 14: crisis_train=1473 (10.3%), w_1=4.83
  Horizonte 15: crisis_train=1485 (10.4%), w_1=4.80
  Horizonte 16: crisis_train=1464 (10.3%), w_1=4.86
  Horizonte 17: crisis_train=1465 (10.3%), w_1=4.86
  Horizonte 18: crisis_train=1438 (10.1%), w_1=4.95

Iniciando entrenamiento con sample weights...
Epoch 1/100
442/446 ━━━━━━━━━━━━━━━━━

2026/07/15 22:29:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/15 22:29:21 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run entrenamiento_cnn_20260715_222731 at: http://localhost:5000/#/experiments/4/runs/f64b8c40822344baa687a99aaf659b22
🧪 View experiment at: http://localhost:5000/#/experiments/4


## Evaluación del modelo

In [12]:
# Evaluar en conjunto de test
y_pred_proba_list = modelo_cnn.predict(X_test)

# Métricas para el primer horizonte (1 mes)
y_pred_1m = (y_pred_proba_list[0] > 0.5).astype(int).flatten()
y_test_1m = y_test_list[0]

test_acc = accuracy_score(y_test_1m, y_pred_1m)
test_prec = precision_score(y_test_1m, y_pred_1m, zero_division=0)
test_recall = recall_score(y_test_1m, y_pred_1m, zero_division=0)

print("=" * 60)
print("MÉTRICAS - Horizonte 1 mes")
print("=" * 60)
print(f"Accuracy: {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall: {test_recall:.4f}")

if len(np.unique(y_test_1m)) > 1:
    try:
        auc_score = roc_auc_score(y_test_1m, y_pred_proba_list[0].flatten())
        print(f"AUC-ROC: {auc_score:.4f}")
    except Exception as e:
        print(f"No se pudo calcular AUC-ROC: {e}")

print("\nReporte detallado:")
print(classification_report(y_test_1m, y_pred_1m))

print("Matriz de confusión:")
print(confusion_matrix(y_test_1m, y_pred_1m))

239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
MÉTRICAS - Horizonte 1 mes
Accuracy: 0.8374
Precision: 0.0000
Recall: 0.0000
AUC-ROC: 0.5000

Reporte detallado:
              precision    recall  f1-score   support

           0       0.84      1.00      0.91      6390
           1       0.00      0.00      0.00      1241

    accuracy                           0.84      7631
   macro avg       0.42      0.50      0.46      7631
weighted avg       0.70      0.84      0.76      7631

Matriz de confusión:
[[6390    0]
 [1241    0]]


/home/ovelez/Documentos/cursos/Maestria/IADegreeThesis/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ovelez/Documentos/cursos/Maestria/IADegreeThesis/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ovelez/Documentos/cursos/Maestria/IADegreeThesis/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` 

In [ ]:

path_plots = os.path.join(PATH_OUTPUT, 'plots')
os.makedirs(path_plots, exist_ok=True)

build_all_plots(
    historia=historia,
    output_dir=path_plots
)

GENERANDO PLOTS INDIVIDUALES DEL MODELO CNN
Generadas 2 imagenes individuales


['output/modelos_cnn/01_loss.png', 'output/modelos_cnn/02_accuracy.png']

## Guardado de artefactos

In [14]:
# Guardar modelo
modelo_path = os.path.join(PATH_OUTPUT, 'modelo_cnn_multi_18m.keras')
modelo_cnn.save(modelo_path)
print(f"Modelo guardado en: {modelo_path}")

# Guardar scaler
scaler_path = os.path.join(PATH_OUTPUT, 'scaler_multi_18m.pkl')
joblib.dump(scaler, scaler_path)
print(f"Scaler guardado en: {scaler_path}")

# Guardar configuración y métricas
config = {
    'ventana_cnn': VENTANA_CNN,
    'max_horizonte': MAX_HORIZONTE,
    'features_numericas': features_numericas,
    'bloques_validos': bloques_validos,
    'metricas_finales': {
        'accuracy': float(test_acc),
        'precision': float(test_prec),
        'recall': float(test_recall),
    },
    'fecha_entrenamiento': datetime.now().isoformat(),
    'num_muestras_train': int(X_train.shape[0]),
    'num_muestras_test': int(X_test.shape[0]),
}

config_path = os.path.join(PATH_OUTPUT, 'config_cnn_18m.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, default=str)
print(f"Configuración guardada en: {config_path}")

print("=" * 60)
print("ARTEFACTOS GUARDADOS")
print("=" * 60)
print(f"1. Modelo: {modelo_path}")
print(f"2. Scaler: {scaler_path}")
print(f"3. Configuración: {config_path}")
print(f"4. Gráfica: {os.path.join(PATH_OUTPUT, 'training_history.png')}")

Modelo guardado en: output/modelos_cnn/modelo_cnn_multi_18m.keras
Scaler guardado en: output/modelos_cnn/scaler_multi_18m.pkl
Configuración guardada en: output/modelos_cnn/config_cnn_18m.json
ARTEFACTOS GUARDADOS
1. Modelo: output/modelos_cnn/modelo_cnn_multi_18m.keras
2. Scaler: output/modelos_cnn/scaler_multi_18m.pkl
3. Configuración: output/modelos_cnn/config_cnn_18m.json
4. Gráfica: output/modelos_cnn/training_history.png


## Resumen de ejecución

- Fecha de ejecución:
- Bloques válidos:
- Muestras de entrenamiento:
- Muestras de prueba:
- Métricas finales:
  - Accuracy:
  - Precision:
  - Recall:
- Artefactos generados:
  - output/modelos_cnn/modelo_cnn_multi_18m.keras
  - output/modelos_cnn/scaler_multi_18m.pkl
  - output/modelos_cnn/config_cnn_18m.json
  - output/modelos_cnn/training_history.png

![icon](../../DocumentosBase/yachayCuadrado.jpg)<br/>***<omar.velez@yachaytech.edu.ec>***<br/>*julio 2026*